# ✨ AI Powered Fashion Design Generator

> **BCA Internship Capstone Project** — runs fully in Google Colab (free tier).

### What does this notebook do?
1. Installs the required libraries (`google-generativeai` and `gradio`).
2. Asks you to paste your free **Gemini API key**.
3. Launches an interactive fashion design generator powered by Google Gemini AI.
4. A public Gradio link is printed — open it in any browser, even on mobile!

### Get your free API key
1. Go to [Google AI Studio](https://aistudio.google.com/app/apikey).
2. Click **Create API key**.
3. Copy the key and paste it in **Cell 2** below.

---

In [ ]:
# ============================================================
# Cell 1 — Install Dependencies
# Run this cell ONCE at the start of every Colab session.
# ============================================================
!pip install -q google-generativeai gradio
print('✅ Libraries installed successfully!')

In [ ]:
# ============================================================
# Cell 2 — Set Your Gemini API Key
# Paste your key between the quotes below, then run this cell.
# ============================================================
import os

# ⬇️  PASTE YOUR API KEY HERE  ⬇️
os.environ['GEMINI_API_KEY'] = 'YOUR_GEMINI_API_KEY_HERE'

# Quick check
if os.environ.get('GEMINI_API_KEY', '').startswith('YOUR_'):
    print('⚠️  Replace YOUR_GEMINI_API_KEY_HERE with your actual API key!')
else:
    print('✅ API key is set. Proceed to Cell 3.')

In [ ]:
# ============================================================
# Cell 3 — Full Application Code
# Run this cell to start the Fashion Design Generator.
# A public URL will appear below — click it to open the app.
# ============================================================

import os
import re
import google.generativeai as genai
import gradio as gr

# ── Configure Gemini model ─────────────────────────────────────────────────
def configure_model():
    api_key = os.environ.get('GEMINI_API_KEY', '')
    if not api_key:
        raise EnvironmentError(
            'GEMINI_API_KEY is not set. Run Cell 2 first.'
        )
    genai.configure(api_key=api_key)
    return genai.GenerativeModel('gemini-1.5-flash')

# ── Input validation ───────────────────────────────────────────────────────
def validate_inputs(gender, occasion, style, garment, color, fabric):
    fields = {
        'Gender': gender, 'Occasion': occasion, 'Fashion Style': style,
        'Garment Type': garment, 'Preferred Color': color, 'Fabric': fabric,
    }
    missing = [n for n, v in fields.items() if not v or v.strip() == '']
    if missing:
        missing_str = ', '.join(missing)
        return False, f'⚠️ Please fill in: {missing_str}'
    return True, ''

# ── Prompt builder ─────────────────────────────────────────────────────────
def build_prompt(gender, occasion, style, garment, color, fabric):
    return f"""
You are a professional and creative fashion designer with 20 years of experience.
A client has given you the following preferences:
- Gender          : {gender}
- Occasion        : {occasion}
- Fashion Style   : {style}
- Garment Type    : {garment}
- Preferred Color : {color}
- Fabric          : {fabric}

Generate a detailed, unique, and wearable fashion design recommendation.
Structure your response EXACTLY as shown below:

**Design Name:** <creative name>
**Design Description:** <2-3 sentences>
**Suggested Silhouette:** <shape/cut of the garment>
**Recommended Fabric:** <specific fabric + reason>
**Color Combination:** <primary + accent colors + rationale>
**Styling Suggestions:** <3-4 practical tips>
**Accessories:** <3-4 accessory recommendations>
**Footwear Suggestions:** <2-3 footwear options>

Keep the language warm and inspiring.
""".strip()

# ── Response parser ────────────────────────────────────────────────────────
SECTION_KEYS = [
    'design name', 'design description', 'suggested silhouette',
    'recommended fabric', 'color combination', 'styling suggestions',
    'accessories', 'footwear suggestions',
]

def parse_response(text):
    results = {k: '' for k in SECTION_KEYS}
    pattern = re.compile(
        r'\*{0,2}(' + '|'.join(re.escape(k) for k in SECTION_KEYS) + r')\*{0,2}\s*:',
        re.IGNORECASE,
    )
    parts = pattern.split(text)
    i = 1
    while i < len(parts) - 1:
        heading = parts[i].strip().lower()
        content = parts[i + 1].strip().strip('*').strip()
        if heading in results:
            results[heading] = content
        i += 2
    return results

# ── Main generation function ───────────────────────────────────────────────
def generate_fashion_design(gender, occasion, style, garment, color, fabric):
    is_valid, error_msg = validate_inputs(gender, occasion, style, garment, color, fabric)
    if not is_valid:
        return (error_msg,) + ('',) * len(SECTION_KEYS)
    try:
        model  = configure_model()
        prompt = build_prompt(gender, occasion, style, garment, color, fabric)
        resp   = model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(temperature=0.85, max_output_tokens=1024),
        )
        sections = parse_response(resp.text)
        status   = '✅ Your personalized fashion design has been generated successfully!'
        return (status,) + tuple(sections[k] for k in SECTION_KEYS)
    except EnvironmentError as e:
        return (f'🔑 API Key Error: {e}',) + ('',) * len(SECTION_KEYS)
    except Exception as e:
        return (f'❌ Error: {e}',) + ('',) * len(SECTION_KEYS)

# ── Gradio UI ──────────────────────────────────────────────────────────────
custom_css = """
.gradio-container { background: linear-gradient(135deg,#1a1a2e,#16213e,#0f3460); font-family:'Segoe UI',system-ui,sans-serif; }
.app-header { background:linear-gradient(90deg,#e94560,#c94fa0,#9b59b6); border-radius:16px; padding:28px 24px; text-align:center; margin-bottom:8px; }
.app-header h1 { color:#fff; font-size:2rem; font-weight:700; margin:0 0 6px; }
.app-header p  { color:rgba(255,255,255,0.85); font-size:1rem; margin:0; }
.generate-btn  { background:linear-gradient(90deg,#e94560,#c94fa0)!important; border:none!important; color:#fff!important; font-size:1.1rem!important; font-weight:600!important; border-radius:12px!important; padding:14px 0!important; }
textarea,.gr-textbox textarea { background:rgba(255,255,255,0.05)!important; color:#f0f0f0!important; border:1px solid rgba(255,255,255,0.15)!important; border-radius:10px!important; }
.status-box textarea { color:#7effa1!important; font-weight:600!important; }
label span { color:#d0d0e8!important; font-weight:500!important; }
"""

gender_opts   = ['Women','Men','Unisex','Non-binary']
occasion_opts = ['Casual Day Out','Office / Workwear','Formal / Black-tie','Wedding / Festive','Beach / Vacation','Party / Night Out','Sports / Active','Date Night']
style_opts    = ['Minimalist','Bohemian','Classic Elegance','Streetwear','Vintage / Retro','Romantic','Edgy / Avant-garde','Preppy','Athleisure','Cottagecore']
garment_opts  = ['Dress','Suit / Blazer','Top + Trousers','Saree / Ethnic Wear','Jumpsuit / Romper','Skirt + Top','Coat / Outerwear','Kurta / Salwar Set','Casual T-shirt + Jeans','Co-ord Set']
color_opts    = ['Classic Black','Pure White','Navy Blue','Emerald Green','Blush Pink','Burgundy / Wine','Mustard Yellow','Terracotta','Lavender','Camel / Nude','Cobalt Blue','Coral']
fabric_opts   = ['Cotton','Silk','Linen','Chiffon','Velvet','Denim','Satin','Georgette','Wool / Tweed','Organza']

with gr.Blocks(css=custom_css, title='AI Fashion Design Generator') as demo:
    gr.HTML('<div class="app-header"><h1>✨ AI Powered Fashion Design Generator</h1><p>Enter your style preferences and let AI craft a personalized design just for you.</p></div>')
    gr.HTML('<div style="color:#e94560;font-weight:600;font-size:1.05rem;margin:12px 0 8px;">🎨 Your Style Preferences</div>')

    with gr.Row():
        g_in  = gr.Dropdown(choices=gender_opts,   label='👤 Gender',         info='Who is this design for?',           interactive=True)
        oc_in = gr.Dropdown(choices=occasion_opts, label='🎉 Occasion',        info='Where will this outfit be worn?',   interactive=True)
        st_in = gr.Dropdown(choices=style_opts,    label='💃 Fashion Style',   info='What style resonates with you?',    interactive=True)
    with gr.Row():
        ga_in = gr.Dropdown(choices=garment_opts,  label='👗 Garment Type',    info='What type of clothing do you want?',interactive=True)
        co_in = gr.Dropdown(choices=color_opts,    label='🎨 Preferred Color', info='Pick your favourite base colour.',  interactive=True)
        fa_in = gr.Dropdown(choices=fabric_opts,   label='🧵 Fabric',          info='What fabric do you prefer?',        interactive=True)

    btn = gr.Button('✨ Generate My Fashion Design', elem_classes=['generate-btn'])

    status_out = gr.Textbox(label='Status', interactive=False, elem_classes=['status-box'], lines=2)

    gr.HTML('<div style="color:#e94560;font-weight:600;font-size:1.05rem;margin:18px 0 8px;">👗 Your Personalized Fashion Design</div>')

    with gr.Row():
        name_out = gr.Textbox(label='🏷️ Design Name',         lines=2, interactive=False)
        sil_out  = gr.Textbox(label='📐 Suggested Silhouette', lines=2, interactive=False)
    desc_out = gr.Textbox(label='📝 Design Description', lines=4, interactive=False)
    with gr.Row():
        fab_out = gr.Textbox(label='🧵 Recommended Fabric',  lines=3, interactive=False)
        col_out = gr.Textbox(label='🎨 Color Combination',   lines=3, interactive=False)
    sty_out = gr.Textbox(label='💡 Styling Suggestions', lines=4, interactive=False)
    with gr.Row():
        acc_out  = gr.Textbox(label='💍 Accessories',         lines=4, interactive=False)
        foot_out = gr.Textbox(label='👠 Footwear Suggestions', lines=4, interactive=False)

    gr.HTML('<div style="margin-top:20px;padding:14px 18px;background:rgba(255,255,255,0.04);border:1px solid rgba(255,255,255,0.1);border-radius:12px;color:#aaa;font-size:0.88rem;"><strong style="color:#e94560;">💡 Tips:</strong> Select all 6 preferences for the most personalised design. Get your free API key at <a href="https://aistudio.google.com/app/apikey" target="_blank" style="color:#e94560;">Google AI Studio</a>.</div>')

    btn.click(
        fn=generate_fashion_design,
        inputs=[g_in, oc_in, st_in, ga_in, co_in, fa_in],
        outputs=[status_out, name_out, desc_out, sil_out, fab_out, col_out, sty_out, acc_out, foot_out],
    )

# Launch with share=True so a public URL is generated (needed in Colab)
demo.launch(share=True, show_error=True)
print('\n✅ App launched! Click the public URL above to open the app.')

---
## 🛑 How to Stop the App
Click **Runtime → Interrupt execution** or press `Ctrl + M + I` in Colab.

## 🔄 How to Restart
Simply run **Cell 3** again — no need to reinstall libraries or reset the key.

## 🔑 Colab Secrets (recommended for reuse)
1. Click the 🔑 icon in the left sidebar.
2. Add a new secret named `GEMINI_API_KEY` with your key as the value.
3. Replace the key assignment in Cell 2 with:
   ```python
   from google.colab import userdata
   os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
   ```
